In [1]:

# ============================================================
# BANKING CUSTOMER BEHAVIOUR & FINANCIAL PERFORMANCE ANALYSIS
# SQL SERVER vs PYTHON VALIDATION
# ============================================================

import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

# ------------------------------------------------------------
# 1. DATABASE CONNECTION
# ------------------------------------------------------------

server = r"DESKTOP-TL9ULDS\SQLEXPRESS"
database = "Banking_Analytics"
driver = "ODBC Driver 18 for SQL Server"

connection_url = URL.create(
    "mssql+pyodbc",
    host=server,
    database=database,
    query={
        "driver": driver,
        "trusted_connection": "yes",
        "TrustServerCertificate": "yes"
    }
)

engine = create_engine(connection_url)

In [2]:
# ------------------------------------------------------------
# 2. LOAD CLEANED DATASETS
# ------------------------------------------------------------

with engine.connect() as connection:

    bank_df = pd.read_sql_query(
        "SELECT * FROM dbo.Bank_Clean",
        connection
    )

    customer_df = pd.read_sql_query(
        "SELECT * FROM dbo.Customer_Clean",
        connection
    )

    transaction_df = pd.read_sql_query(
        "SELECT * FROM dbo.Transactions_Clean",
        connection
    )


In [3]:
# ------------------------------------------------------------
# 3. PYTHON METRICS
# ------------------------------------------------------------

python_metrics = {

    # Banking metrics
    "Total Branches": len(bank_df),

    "Available Revenue Records":
        bank_df["Firm_Revenue"].count(),

    "Total Revenue":
        bank_df["Firm_Revenue"].sum(),

    "Total Expenses":
        bank_df["Expenses"].sum(),

    # Customer metrics
    "Total Customers":
        customer_df["Customer_ID"].count(),

    "Unique Customers":
        customer_df["Customer_ID"].nunique(),

    # Transaction metrics
    "Total Transactions":
        transaction_df["Transaction_ID"].count(),

    "Unique Transactions":
        transaction_df["Transaction_ID"].nunique(),

    "Total Transaction Amount":
        transaction_df["Transaction_Amount"].sum(),

    # Investment metrics
    "Investment Records":
        transaction_df["Investment_Amount"].count(),

    "Recorded Investment Amount":
        transaction_df["Investment_Amount"].sum()
}

In [4]:
# ------------------------------------------------------------
# 4. SQL SERVER METRICS
# ------------------------------------------------------------

sql_query = """
SELECT
    (SELECT COUNT(*)
     FROM dbo.Bank_Clean) AS Total_Branches,

    (SELECT COUNT(Firm_Revenue)
     FROM dbo.Bank_Clean) AS Available_Revenue_Records,

    (SELECT SUM(Firm_Revenue)
     FROM dbo.Bank_Clean) AS Total_Revenue,

    (SELECT SUM(Expenses)
     FROM dbo.Bank_Clean) AS Total_Expenses,

    (SELECT COUNT(Customer_ID)
     FROM dbo.Customer_Clean) AS Total_Customers,

    (SELECT COUNT(DISTINCT Customer_ID)
     FROM dbo.Customer_Clean) AS Unique_Customers,

    (SELECT COUNT(Transaction_ID)
     FROM dbo.Transactions_Clean) AS Total_Transactions,

    (SELECT COUNT(DISTINCT Transaction_ID)
     FROM dbo.Transactions_Clean) AS Unique_Transactions,

    (SELECT SUM(Transaction_Amount)
     FROM dbo.Transactions_Clean) AS Total_Transaction_Amount,

    (SELECT COUNT(Investment_Amount)
     FROM dbo.Transactions_Clean) AS Investment_Records,

    (SELECT SUM(Investment_Amount)
     FROM dbo.Transactions_Clean) AS Recorded_Investment_Amount
"""

with engine.connect() as connection:

    sql_result = connection.exec_driver_sql(
        sql_query
    ).mappings().one()

engine.dispose()

In [5]:

# ------------------------------------------------------------
# 5. PREPARE SQL METRICS
# ------------------------------------------------------------

sql_metrics = {

    "Total Branches":
        sql_result["Total_Branches"],

    "Available Revenue Records":
        sql_result["Available_Revenue_Records"],

    "Total Revenue":
        sql_result["Total_Revenue"],

    "Total Expenses":
        sql_result["Total_Expenses"],

    "Total Customers":
        sql_result["Total_Customers"],

    "Unique Customers":
        sql_result["Unique_Customers"],

    "Total Transactions":
        sql_result["Total_Transactions"],

    "Unique Transactions":
        sql_result["Unique_Transactions"],

    "Total Transaction Amount":
        sql_result["Total_Transaction_Amount"],

    "Investment Records":
        sql_result["Investment_Records"],

    "Recorded Investment Amount":
        sql_result["Recorded_Investment_Amount"]
}


In [6]:
# ------------------------------------------------------------
# 6. COMPARE SQL AND PYTHON RESULTS
# ------------------------------------------------------------

comparison_df = pd.DataFrame({

    "Metric": list(python_metrics.keys()),

    "SQL_Result": [
        sql_metrics[key]
        for key in python_metrics.keys()
    ],

    "Python_Result": [
        python_metrics[key]
        for key in python_metrics.keys()
    ]

})

# Convert numeric results to comparable values
comparison_df["SQL_Result"] = pd.to_numeric(
    comparison_df["SQL_Result"]
)

comparison_df["Python_Result"] = pd.to_numeric(
    comparison_df["Python_Result"]
)

comparison_df["Difference"] = (
    comparison_df["SQL_Result"] -
    comparison_df["Python_Result"]
)

comparison_df["Validation_Status"] = np.isclose(
    comparison_df["SQL_Result"].fillna(0),
    comparison_df["Python_Result"].fillna(0),
    rtol=1e-9,
    atol=0.01
)

comparison_df["Validation_Status"] = (
    comparison_df["Validation_Status"]
    .map({
        True: "MATCHED",
        False: "MISMATCHED"
    })
)


In [7]:
# ------------------------------------------------------------
# 7. DISPLAY VALIDATION RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SQL SERVER vs PYTHON VALIDATION RESULTS")
print("=" * 70)

display(comparison_df.round(2))


SQL SERVER vs PYTHON VALIDATION RESULTS


,Metric,SQL_Result,Python_Result,Difference,Validation_Status
0,Total Branches,1.000000e+03,1.000000e+03,0.0,MATCHED
1,Available Revenue Records,9.500000e+02,9.500000e+02,0.0,MATCHED
2,Total Revenue,4.883705e+08,4.883705e+08,0.0,MATCHED
3,Total Expenses,2.653260e+08,2.653260e+08,0.0,MATCHED
4,Total Customers,1.000000e+04,1.000000e+04,0.0,MATCHED
5,Unique Customers,1.000000e+04,1.000000e+04,0.0,MATCHED
6,Total Transactions,1.000000e+04,1.000000e+04,0.0,MATCHED
7,Unique Transactions,1.000000e+04,1.000000e+04,0.0,MATCHED
8,Total Transaction Amount,2.542708e+07,2.542708e+07,0.0,MATCHED
9,Investment Records,1.000000e+04,1.000000e+04,0.0,MATCHED


In [8]:
# ------------------------------------------------------------
# 8. VALIDATION SUMMARY
# ------------------------------------------------------------

matched = (
    comparison_df["Validation_Status"] == "MATCHED"
).sum()

mismatched = (
    comparison_df["Validation_Status"] == "MISMATCHED"
).sum()

print("\n" + "=" * 70)
print("FINAL VALIDATION SUMMARY")
print("=" * 70)

print("Total Metrics Validated:", len(comparison_df))
print("Matched Metrics:", matched)
print("Mismatched Metrics:", mismatched)

if mismatched == 0:
    print("\nAll selected SQL and Python metrics match!")
else:
    print("\nSome metrics require further investigation.")

print("\nSQL-Python Validation Completed!")



FINAL VALIDATION SUMMARY
Total Metrics Validated: 11
Matched Metrics: 11
Mismatched Metrics: 0

All selected SQL and Python metrics match!

SQL-Python Validation Completed!
